# Day 62 — Model Evaluation Metrics and K-Fold Cross-Validation

This notebook covers Model Evaluation (MAE, MSE, RMSE, $R^2$ Score), Outlier Sensitivity, Flaws of Single Train/Test Splits, and K-Fold Cross-Validation

## Validation goals
- Master industry-standard regression evaluation metrics: Mean Absolute Error, Mean Squared Error, Root Mean Squared Error, and Coefficient of Determination ($R^2$)
- Analyze how squaring errors in MSE amplifies outlier penalties and affects gradient-based optimization
- Recognize the limitations and high variance risks of a single random train/test split
- Implement K-Fold Cross-Validation in Python using Scikit-learn (`cross_val_score`) to secure robust, unbiased performance estimates

When building models that predict continuous numbers (like house prices or sales forecasts), you cannot use classification metrics. You need mathematical tools that measure numerical error, alongside robust techniques to ensure your evaluation isn't just a stroke of luck.

### 1. Regression Evaluation Metrics

When evaluating a regression model, you want to know: How far off are our predictions from the true numerical values? Here are the four industry-standard metrics used in Scikit-learn:

**A. Mean Absolute Error (MAE)**

MAE calculates the average absolute distance between every prediction and the actual value.$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \vert{}y_i - \hat{y}_i$$
- Why it's great: It is highly interpretable. If your MAE is 15.5 on a house price model, it means your model's predictions are off by an average of $\$15,500$.

**B. Mean Squared Error (MSE)**

MSE squares every error before averaging them out.$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
-Why it matters: Because errors are squared, large mistakes are heavily penalized. If your model makes one massive blunder, the MSE will shoot up drastically.

**C. Root Mean Squared Error (RMSE)**

RMSE is simply the square root of the MSE.$$\text{RMSE} = \sqrt{\text{MSE}}$$
- Why it's used: MSE squares the error units (e.g., dollars squared), making it hard to interpret directly. Taking the square root brings the error metric back into the exact same units as your target variable.

**D. $R^2$ Score (Coefficient of Determination)**

Unlike MAE and RMSE (which measure raw error in units), the $R^2$ score represents the proportion of variance in the dependent variable that is predictable from the features.
- A score of $1.0$ means a 100% perfect fit.
- A score of $0.0$ means your model performs no better than a simple horizontal baseline (just guessing the average value).
- A negative score means your model is actively worse than a horizontal baseline.

### 2. The Flaw of a Single Train/Test Split

When you run `train_test_split()`, your data is partitioned randomly. But what if by sheer bad luck, all the hardest, most complex data points landed in your test set, while all the easy data landed in your training set? Your test accuracy would look artificially terrible. Conversely, if easy data landed in your test set, your score would look artificially brilliant.

To fix this, we use K-Fold Cross-Validation.

### 3. K-Fold Cross-Validation

### Concept Explanation

Instead of splitting your data just once, K-Fold Cross-Validation does the following:
- Splits your entire dataset into $K$ equal parts or "folds" (commonly $K = 5$ or $K = 10$).
- Trains the model on $K-1$ folds and tests it on the remaining 1 single fold.
- Repeats this process $K$ times so that every single fold gets a chance to act as the test set.
- Averages all $K$ performance scores together to give you a single, robust, unbiased performance evaluation.

### 4. Implementing Cross-Validation and Metrics in Python

Let's write a clean Python script using Scikit-learn to evaluate a regression model using both test metrics and 5-fold cross-validation.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Setup synthetic regression dataset
np.random.seed(42)
X = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])
y = np.array([3.1, 4.9, 7.2, 8.0, 11.1, 12.0, 14.5, 15.8, 18.2, 20.1])

# 2. Standard Train/Test Split evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Compute metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- Single Train/Test Split Metrics ---")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}\n")

# 3. K-Fold Cross-Validation (cv=5) across the entire dataset
# scoring='r2' measures the R-squared across each fold
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print("--- 5-Fold Cross-Validation ---")
print(f"Individual Fold R² Scores: {np.round(cv_scores, 4)}")
print(f"Mean CV R² Score:          {cv_scores.mean():.4f}")

--- Single Train/Test Split Metrics ---
MAE:  0.1207
RMSE: 0.1421
R²:   0.9995

--- 5-Fold Cross-Validation ---
Individual Fold R² Scores: [ 0.9964 -1.0001 -0.2176  0.7502  0.8351]
Mean CV R² Score:          0.2728


### Guided Question:

Why is Mean Squared Error (MSE) much more sensitive to outlier errors than Mean Absolute Error (MAE)?

### Answer:

Mean Squared Error (MSE) is significantly more sensitive to outliers than Mean Absolute Error (MAE) because of how each metric treats the magnitude of errors.

The Mathematical FormulasTo understand why, let's look at their definitions, where $y_i$ is the true value, $\hat{y}_i$ is the predicted value, and $n$ is the total number of samples:
- Mean Absolute Error (MAE):$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \vert{}y_i - \hat{y}_i\vert{}$$
- MAE calculates the average of the absolute differences between predictions and actual values. The penalty increases linearly with the size of the error.
- Mean Squared Error (MSE):$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

MSE calculates the average of the squared differences. Because the errors are squared, the penalty grows quadratically with the size of the error.

**A Concrete Numerical Example**

Suppose a model makes an error on a standard data point versus an outlier:
- Standard Error of $2$:
    - MAE Contribution: $\vert{}2\vert{} = 2$
    - MSE Contribution: $2^2 = 4$
- Large Outlier Error of $10$ (5x larger):
    - MAE Contribution: $\vert{}10\vert{} = 10$ (grows 5x, matching the error increase)
    - MSE Contribution: $10^2 = 100$ (grows 25x, drastically compounding the penalty)

Because MSE squares the residual, a single massive outlier generates a disproportionately gigantic error score, heavily skewing the overall metric.

**The Impact on Model Training (Gradient Optimization)**

This sensitivity isn't just an evaluation quirk; it fundamentally changes how gradient-based machine learning models learn during training:
- Gradient Scaling in MSE: The derivative of the squared error loss with respect to the prediction is $2(y_i - \hat{y}_i)$. The gradient scales linearly with the size of the error. When an outlier is present, the model experiences a massive gradient jolt, forcing the optimization algorithm to bend its weights aggressively to minimize that single outlier, often ruining the fit for the normal data points.
- Constant Gradients in MAE: The derivative of the absolute error is constant ($\pm 1$ depending on the sign of the error). Outliers do not create disproportionately massive gradient updates, making MAE much more robust (resistant) to noisy data and extreme values.

**When to Choose Which**

- Use MSE when large errors are exceptionally dangerous or costly, and you want your model to severely penalize any prediction that strays far from the truth.
- Use MAE (or Huber Loss, which combines both) when your dataset contains known outliers or heavy noise that you do not want to distort the model's overall learning process.

### Guided Question:

Why is 5-fold cross-validation a more reliable gauge of a model's real-world performance than a single train_test_split()?

### Answer:

5-fold cross-validation provides a significantly more robust and reliable gauge of real-world performance than a single `train_test_split()` because it eliminates the "luck of the draw" inherent in random partitioning.

**The Limitations of a Single `train_test_split()`**

When you perform a single train-test split (e.g., an 80/20 split), your evaluation depends entirely on one specific random partition of your data. This introduces major vulnerabilities:
- High Variance (The Lucky Split): If the random seed happens to place easy-to-classify samples in the test set and complex edge cases in the training set, your test accuracy will be artificially inflated. Conversely, a hostile split can make a great model look terrible.
- Data Waste: A standard split discards a portion of your data (e.g., 20%) from the learning process entirely. If your dataset is already small or contains rare classes, withholding data from training starves the model of valuable patterns.
- Single Point Estimate: You get only one accuracy number. You have no visibility into how stable that score is or how sensitive the model is to different subsets of data.

**How 5-Fold Cross-Validation Solves This**

In 5-fold cross-validation, the dataset is systematically partitioned into 5 equal-sized subsets (folds). The training and evaluation process runs 5 distinct iterations:
- Iteration 1: Train on Folds 1, 2, 3, and 4; evaluate on Fold 5.
- Iteration 2: Train on Folds 1, 2, 3, and 5; evaluate on Fold 4....and so on, until every fold has served as the test set exactly once.

This architecture solves the core issues of single-split evaluation:
- Exhaustive Utilization: Every single data point in your dataset is used for both training and validation. Every sample gets tested against a model that was trained without seeing it.
- Aggregated Performance: Instead of relying on one arbitrary score, you compute the mean and standard deviation across all 5 folds. This gives you a confidence interval (e.g., $88.4\% \pm 1.2\%$), revealing true generalization stability.
- Resistance to Data Skew: If one fold happens to contain weird outliers or noise, its impact is diluted because four other independent evaluations balance it out.

### Guided Question:

In the Scikit-learn snippet above, what does the cv=5 parameter instruct the cross_val_score() function to do with your dataset?

### Answer:

In the code snippet, the `cv=5` parameter instructs `cross_val_score()` to perform 5-fold cross-validation by dividing the dataset into 5 equal-sized subsets (`folds`).Specifically, it dictates the following execution cycle:
- Partitioning: Divides `X` and `y` into 5 distinct, non-overlapping subsets.
- Iterative Training & Testing: Runs 5 separate training/evaluation loops. In each loop, 4 folds (80% of the data) are combined to train the `LinearRegression` model, and the remaining 1 fold (20%) is held out as the test set to compute the score.
- Output Generation: Returns an array of 5 individual $R^2$ scores (one for each fold), allowing you to evaluate the model's consistency and generalization across different slices of the data rather than relying on a single random split.